# Najafi et al. (2020) on DANDI:000016, part 3: decoder time course and learning

Paper doi:10.1016/j.neuron.2019.09.045; dataset DANDI:000016 version 0.260915.2309, the only
published version (the paper cites none). Claims file `claims/najafi-2020.yaml`.

Decoders as in `02_decoders.ipynb` (coarser C grid, a deviation). For the time courses a decoder
is trained and tested at every 97 ms bin with the C chosen at bin -1 (0-97 ms before the
choice); the paper chooses C at every bin, which was too slow here (a deviation); `04_time_course_per_bin_C.ipynb` repeats fig3b for mice 2 and 3 with C chosen at every bin. The per-bin weights of fig3df are taken from these time-course decoders. Learning mice
are mice 1, 2 and 4 (fni16, fni17, fni19; claims file). Excitatory populations are 3 random
subsamples of as many excitatory neurons as the session has inhibitory neurons (the paper 50), and the time courses use 25 random splits per bin (the paper 50).

| Claim | Here |
|---|---|
| fig3b-decoding-time-course | tested |
| fig3df-weights-similar | per-bin part tested |
| fig8ab-learning-accuracy | tested |
| fig8ce-choice-signal-onset | tested |
| fig8de-learning-parallel | tested |
| figs13c-equal-trials | tested |
| figs13ab-pseudo-learning | tested |

In [ ]:
import time, resource, os
import numpy as np
import pandas as pd
from scipy import stats
from joblib import Parallel, delayed
import najafi as nj

T0 = time.time()
A = nj.assets()
S = nj.load_all(A, workers=6, alignments=(nj.CHOICE,))
for s in S:
    s["al"][nj.CHOICE].pop("frames", None)
print("loaded", len(S), "sessions in %.0f s, peak memory %.1f GB" % (time.time() - T0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
L = pd.DataFrame([{"i": i, "mouse": s["mouse"], "session": s["session_id"], "date": s["start"][:10],
                   "perf": nj.easy_performance(s["trials"])[0], "n_easy": nj.easy_performance(s["trials"])[1],
                   "correct": int(nj.trial_mask(s, (1,)).sum()), "incorrect": int(nj.trial_mask(s, (0,)).sum())} for i, s in enumerate(S)])
L["day"] = L.groupby("mouse").date.rank(method="first").astype(int)
for m, g in L[L.mouse.isin(nj.LEARNING_MICE)].groupby("mouse"):
    el = nj.early_late(g.perf.to_numpy(), g.day.to_numpy())
    for rd in ("a", "b"):
        L.loc[g.index, "early_" + rd] = el[rd][0]; L.loc[g.index, "late_" + rd] = el[rd][1]
    print(f"mouse {m}: {len(g)} days {g.date.min()} to {g.date.max()}; easy-trial performance {g.perf.min():.2f} to {g.perf.max():.2f} "
          f"(r with day {stats.pearsonr(g.day, g.perf)[0]:.2f}); early (a) days {sorted(g.day[el['a'][0]])}, late (a) days {sorted(g.day[el['a'][1]])}; "
          f"early (b) {int(el['b'][0].sum())}, late (b) {int(el['b'][1].sum())} days")
print(L.groupby("mouse")[["correct", "incorrect", "n_easy"]].median())
# fig S13C: the same number of correct trials of each choice in every session of a mouse, the minimum over its sessions
neq = {}
for m in nj.LEARNING_MICE:
    v = [min(np.sum(s["trials"].choice.to_numpy()[nj.trial_mask(s, (1,))] == c) for c in (0, 1)) for s in S if s["mouse"] == m]
    v = [x for x in v if x >= nj.MIN_TRIALS]
    neq[m] = int(min(v))
print("correct trials per choice in every session (fig S13C), minimum over sessions with at least 10:", neq)

In [ ]:
os.environ.setdefault("OMP_NUM_THREADS", "1")
t1 = time.time()
OUT = Parallel(n_jobs=4)(delayed(nj.learn_session)(s, 5000 + i, n_equal=neq.get(s["mouse"]), learning=s["mouse"] in nj.LEARNING_MICE) for i, s in enumerate(S))
print("decoders: %.1f min" % ((time.time() - t1) / 60))
rows = []
for o, (_, l) in zip(OUT, L.iterrows()):
    for rd in ("a", "b"):
        if rd in o:
            for p, r in o[rd].items():
                pop = "exc" if p.startswith("exc") else p
                rows.append({"mouse": o["mouse"], "session": o["session"], "rd": rd, "pop": pop, "sub": p, **{k: r.get(k, np.nan) for k in ("acc", "acc_sh", "on05", "on001", "pseudo")}})
R = pd.DataFrame(rows).groupby(["mouse", "session", "rd", "pop"])[["acc", "acc_sh", "on05", "on001", "pseudo"]].mean().reset_index()
R = R.merge(L[["session", "perf", "day", "early_a", "late_a", "early_b", "late_b"]], on="session")
print("learning sessions decoded, reading (a):", R[(R.rd == "a") & (R["pop"] == "all")].groupby("mouse").size().to_dict(),
      "reading (b):", R[(R.rd == "b") & (R["pop"] == "all")].groupby("mouse").size().to_dict())

## fig3b-decoding-time-course

> Accuracy of the all-neuron choice decoder grows after stimulus onset and peaks at the time of
> the choice.

In [ ]:
TC = pd.DataFrame([{"mouse": o["mouse"], "k": k, "acc": a, "sh": b} for o in OUT if "tc_corr" in o for k, a, b in zip(o["k"], *o["tc_corr"])])
for m, g in TC.groupby("mouse"):
    n = g.groupby("k").size(); nmax = n.max()
    v = g.groupby("k").acc.mean()[n >= 0.5 * nmax]; vs = g.groupby("k").sh.mean()[n >= 0.5 * nmax]
    kp = int(v.idxmax()); first = int(v.index.min())
    print(f"mouse {m} ({nmax} sessions): bins {first} to {int(v.index.max())}; earliest {v.loc[first]:.1f}, bin -1 {v.loc[-1]:.1f}, bin 0 {v.loc[0]:.1f}, "
          f"peak {v.max():.1f} at bin {kp}; (a) peak in bins -1 or 0: {kp in (-1, 0)}; (b) peak within 300 ms (bins -3 to 2): {-3 <= kp <= 2}; "
          f"earliest below bin -1: {v.loc[first] < v.loc[-1]}; shuffle {vs.min():.1f} to {vs.max():.1f}")
    print("   accuracy by bin:", v.round(1).to_dict())

**Result: not reproduced.** With C chosen at bin -1 and 25 splits (03), all-neuron accuracy on correct trials rises from the earliest bin (59.4 to 70.2 percent) to the bin before the choice (72.4 to 86.4) in every mouse; the peak falls at bin 2 in mice 1 and 4, bin 3 in mouse 2 (86.6, equal to bin 2 at one decimal) and bin 6 in mouse 3. With C chosen at every bin and 50 splits (04, mice 2 and 3), mouse 2 peaks at bin 2 (88.4, next 88.2) and mouse 3 at bin 6 (83.6). The peak is never in bins -1 or 0, reading (a), and in mouse 3 it lies outside 300 ms, reading (b), under both C rules, so the claim fails under both readings; the criterion has no partial band.


## fig3df-weights-similar, per bin

> In the decoder trained on all neurons, excitatory and inhibitory neurons receive weights of
> matched magnitude throughout the trial and with very similar distributions.

The criterion asks for the mean |w| per cell type at 0-97 ms before the choice (in
`02_decoders.ipynb`) and per bin. Here the per-bin part: the all-neuron time-course decoders of
fig3b (correct trials, C chosen at bin -1, 25 splits), weights averaged over the splits and
normalised to unit length at each bin, mean |w| per cell type per session, and the unpaired
t-test across sessions per mouse at each bin.

In [ ]:
WB = pd.DataFrame([{"mouse": o["mouse"], "k": k, "E": e, "I": i} for o in OUT if "w_bins" in o for k, e, i in zip(o["k"], *o["w_bins"])])
for m, g in WB.groupby("mouse"):
    n = g.groupby("k").size(); keep = n.index[n >= 0.5 * n.max()]
    res = []
    for k in keep:
        x = g[g.k == k]
        res.append((int(k), x.E.mean(), x.I.mean(), stats.ttest_ind(x.E, x.I).pvalue))
    ps = np.array([r[3] for r in res])
    print(f"mouse {m} ({n.max()} sessions): {len(res)} bins; bins with p < 0.05: {[r[0] for r in res if r[3] < 0.05]}; smallest p {ps.min():.3g}; "
          f"I - E from {min(r[2]-r[1] for r in res):+.4f} to {max(r[2]-r[1] for r in res):+.4f}")
    print("   mean |w| E/I by bin:", {r[0]: f"{r[1]:.4f}/{r[2]:.4f}" for r in res})
print("range of the per-session mean |w|: %.4f to %.4f" % (WB[["E", "I"]].min().min(), WB[["E", "I"]].max().max()))


**Result: partially reproduced.** At 0-97 ms before the choice (02), the mean |w| of unit-length weight vectors of the all-neuron decoder does not differ between types in any mouse across sessions (p = 0.75, 0.051, 0.32, 0.13). Per bin (03, the time-course decoders with C chosen at bin -1), inhibitory weights are larger in 12 of 25 bins of mouse 2 (smallest p 0.0041) and in 1 of 25 bins of mouse 4 (p = 0.012), and no bin differs in mice 1 and 3 (smallest p 0.10 and 0.14). So two mice differ in part of the trial, the criterion's partial band. The pooled counts, 42019 excitatory and 5172 inhibitory neurons, equal the expected values.


## fig8ab-learning-accuracy

> During learning, choice decoding accuracy increases and correlates with behavioural
> performance across sessions, for all, excitatory and inhibitory populations (r = 0.55, 0.35,
> 0.32 in mouse 2 and 0.57, 0.63, 0.32 in mouse 3).

In [ ]:
exp_r = {"legend mouse 2": (0.55, 0.35, 0.32), "legend mouse 3": (0.57, 0.63, 0.32)}
for rd in ("a", "b"):
    print("trials, reading", rd)
    for m in nj.LEARNING_MICE:
        out = []
        for pop in ("all", "exc", "inh"):
            g = R[(R.rd == rd) & (R.mouse == m) & (R["pop"] == pop)].dropna(subset=["acc", "perf"])
            r, p = stats.pearsonr(g.perf, g.acc)
            out.append(f"{pop} r = {r:.2f} (p = {p:.2g}, {len(g)} days)")
        print(f"  mouse {m}: " + "; ".join(out))
print("expected: legend 'mouse 2' r = 0.55, 0.35, 0.32; legend 'mouse 3' r = 0.57, 0.63, 0.32 (tolerance 10%);"
      " reading (a) of the names is the paper's mice 2 and 4, reading (b) the paper's mice 2 and 3 (mouse 3 was not imaged through learning and is not decoded here)")

**Result: not reproduced.** Decoder accuracy correlates positively with easy-trial performance in every learning mouse and population. With all trials, reading (a), mouse 1 has r = 0.66, 0.57, 0.53 (all p < 0.001), mouse 2 0.39, 0.23, 0.20 (p = 0.0067, 0.13, 0.19) and mouse 4 0.57, 0.56, 0.31 (p = 0.00035, 0.00042, 0.068). Reading (b) gives mouse 2 0.36, 0.06, 0.20 and mouse 4 0.29, 0.27, 0.11. Under the reading of the legend's mice as the paper's mice 2 and 4, mouse 2 matches none of the reported values and mouse 4 matches 0.57 and 0.32 but not 0.63 (0.56). Not all p < 0.05, so neither the reproduced nor the partial band holds.


## fig8ce-choice-signal-onset

> The choice signal emerges progressively earlier in the trial as mice become experts, so the
> onset of above-chance decoding correlates negatively with performance (r = -0.60, -0.34, -0.38
> in mouse 2 and -0.60, -0.27, -0.28 in mouse 3 for all, excitatory and inhibitory).

In [ ]:
for col, lab in (("on05", "p < 0.05 (Methods)"), ("on001", "variant p < 0.001 (code)")):
    for rd in ("a", "b"):
        print(lab, "trials reading", rd)
        for m in nj.LEARNING_MICE:
            out = []
            for pop in ("all", "exc", "inh"):
                g = R[(R.rd == rd) & (R.mouse == m) & (R["pop"] == pop)].dropna(subset=[col, "perf"])
                if len(g) > 2:
                    r, p = stats.pearsonr(g.perf, g[col])
                    out.append(f"{pop} r = {r:.2f} (p = {p:.2g}, {len(g)} days with onset, median onset {g[col].median():.0f} ms)")
            print(f"  mouse {m}: " + "; ".join(out))

**Result: not reproduced.** The onset of the first 500 ms run of above-shuffle accuracy correlates negatively with performance in mouse 1 for the all-neuron and excitatory decoders (r = -0.49, -0.41; p = 0.0013, 0.014), not for inhibitory (r = -0.07). In mice 2 and 4 it does not (r -0.26 to 0.08, p >= 0.12), under trial reading (a) and the Methods alpha 0.05. Reading (b) and the code's alpha 0.001 give no more. None of the six reported values is matched. Per-bin C was not run for these decoders (estimated over 6 hours of CPU).


## fig8de-learning-parallel

> The choice signal emerges at the same time in excitatory and inhibitory populations, and its
> magnitude and timing are matched for the two cell types throughout learning.

In [ ]:
def tt(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float); a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    return stats.ttest_ind(a, b).pvalue if len(a) > 1 and len(b) > 1 else np.nan
for rd in ("a", "b"):
    for el in ("a", "b"):
        print(f"trials reading {rd}, early/late reading {el}")
        for m in nj.LEARNING_MICE:
            g = R[(R.rd == rd) & (R.mouse == m)]
            out = []
            for per in ("early", "late"):
                gg = g[g[per + "_" + el].astype(bool)]
                e, i = gg[gg["pop"] == "exc"], gg[gg["pop"] == "inh"]
                out.append(f"{per} ({len(i)} days): acc exc {e.acc.mean():.1f} inh {i.acc.mean():.1f} p = {tt(e.acc, i.acc):.2g}; "
                           f"onset exc {e.on05.mean():.0f} inh {i.on05.mean():.0f} ms p = {tt(e.on05, i.on05):.2g}")
            le = []
            for pop in ("exc", "inh"):
                x = g[g["pop"] == pop]
                lt, er = x[x["late_" + el].astype(bool)].acc, x[x["early_" + el].astype(bool)].acc
                le.append(f"{pop} late {lt.mean():.1f} vs early {er.mean():.1f} p = {tt(lt, er):.2g}")
            print(f"  mouse {m}: " + " | ".join(out) + " | " + "; ".join(le))

**Result: not reproduced.** With all trials, reading (a), and early and late days by reading (a), the subsampled excitatory and inhibitory decoders do not differ in accuracy or onset in early or late days in any learning mouse (p 0.22 to 0.91 where testable). Late accuracy exceeds early for both types in mice 1 and 4 (p 0.0013 to 0.032), but not in mouse 2 (p = 0.07, 0.26). Under trial reading (b), late-day accuracy differs between types in mouse 2 (I 67.6 against E 60.5, p = 0.024), and late exceeds early for excitatory neurons in neither mouse 2 nor mouse 4. Early and late days by reading (b) give 0 or 1 day in most mice and cannot be tested. The claim fails under both trial readings, so it is not reproduced.


## figs13c-equal-trials

> The learning-related increase in decoding accuracy remains when the number of trials is equal
> in every session.

In [ ]:
EQ = pd.DataFrame([{"session": o["session"], "mouse": o["mouse"], **o["equal"]} for o in OUT if "equal" in o]).merge(L[["session", "early_a", "late_a", "early_b", "late_b"]], on="session")
for el in ("a", "b"):
    for m, g in EQ.groupby("mouse"):
        e, l = g[g["early_" + el].astype(bool)].acc, g[g["late_" + el].astype(bool)].acc
        print(f"early/late reading {el}, mouse {m}: {g.n.iloc[0]} trials per choice, {len(g)} sessions; early {e.mean():.1f} ({len(e)}), late {l.mean():.1f} ({len(l)}), "
              f"late - early {l.mean()-e.mean():+.1f}, p = {tt(l, e):.2g}")

**Result: not reproduced.** With 12, 14 and 15 correct trials of each choice in every session, all-neuron accuracy is higher late than early in every learning mouse (+7.4, +4.4 and +16.7 points), significantly only in mouse 4 (p = 0.0046), not in mice 1 and 2 (p = 0.073, 0.32).


## figs13ab-pseudo-learning

> Decoding accuracy of pseudo populations, which lack noise correlations, also increases
> significantly with training, so reduced noise correlations do not alone explain the improved
> accuracy.

In [ ]:
for el in ("a", "b"):
    for m in nj.LEARNING_MICE:
        g = R[(R.rd == "a") & (R.mouse == m)]
        out = []
        for pop in ("all", "exc", "inh"):
            x = g[g["pop"] == pop]
            e, l = x[x["early_" + el].astype(bool)].pseudo, x[x["late_" + el].astype(bool)].pseudo
            out.append(f"{pop} early {e.mean():.1f} late {l.mean():.1f} p = {tt(l, e):.2g}")
        print(f"early/late reading {el}, mouse {m}: " + "; ".join(out))

**Result: not reproduced.** Pseudo-population accuracy of the all-neuron decoder rises from early to late days in every learning mouse (75.5 to 95.5, 80.5 to 88.3, 73.3 to 92.9). The rise is significant in mice 1 and 4 (p = 0.00025, 0.0011) and not in mouse 2 (p = 0.18). Excitatory and inhibitory pseudo populations rise in the same direction in every mouse.


## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| data-composition | reproduced |
| fig1b-psychometric | reproduced |
| figs1b-trial-history | blocked, data missing |
| fig1gi-inferred-spiking-differs | not reproduced |
| fig2a-example-session | not reproduced |
| fig2bc-fraction-selective-similar | not reproduced |
| fig2bd-selectivity-time-course | blocked, methods underspecified |
| fig2de-selectivity-magnitude-similar | partially reproduced |
| figs3d-same-spiking-selectivity | reproduced |
| figs3c-wider-window | not reproduced |
| figs3e-correct-error-correlation | not reproduced |
| fig3bc-decoding-similar | reproduced |
| fig3b-decoding-time-course | not reproduced |
| fig3df-weights-similar | partially reproduced |
| fig3hi-mixed-no-advantage | reproduced |
| figs5-fluorescence-decoding | blocked, data missing |
| figs6ab-previous-choice-stimulus | blocked, other |
| figs6c-outcome-decoding | blocked, other |
| figs6de-incorrect-trials | not reproduced |
| fig4bd-stability-similar | reproduced |
| figs7c-angle-stability | blocked, too large |
| fig5a-nonselective-model, fig5bc-selective-models-match, figs9-signal-selective-best | blocked, not reached in the run's time (simulations) |
| fig6a-same-selectivity-correlations | reproduced |
| fig6bc-inhibitory-correlations-higher | blocked, methods underspecified |
| fig7-noise-correlations-limit-decoding | blocked, methods underspecified |
| fig8ab-learning-accuracy | not reproduced |
| fig8ce-choice-signal-onset | not reproduced |
| fig8de-learning-parallel | not reproduced |
| figs13c-equal-trials | not reproduced |
| figs12-licking-running | not reproduced |
| fig8f-correlations-decrease | partially reproduced |
| figs13ab-pseudo-learning | not reproduced |
| fig8g-fraction-selective-triples | not reproduced |
| figs2-temporal-dynamics-similar | blocked, other |
| figs14-neuropil, figs15-labeling-specificity | blocked, data missing (raw movies, histology) |

How the analysis differs from the paper's: the SVM regularisation is chosen from a grid with steps
of 1 in log10 (the paper 0.2; the check in `02_decoders.ipynb` shows differences of at most 0.67
points); excitatory, mixed and 2n-excitatory populations use 5 subsamples (3 in the learning
analysis) where the paper uses 50; the time courses use 25 splits per bin and one C per session
chosen at the bin before the choice; choices are matched once per session; and side labels are
"high-rate" and "low-rate" choices, as the files do not name the side. 16 Hz trials are left out everywhere, as their choice side is unknown; the trial table has no flag for helped trials or the first trials of a session, so those exclusions of the code are not applied; notebook 04 repeats the fig3b time course for mice 2 and 3 with C chosen at every bin and 50 splits.

What was found about the dataset: the session counts, trials per session and neuron counts match
the paper; the traces named `dFoF_*` hold the normalised inferred spiking activity, not Delta F/F,
and no fluorescence trace is in the files; the attribute named `rate` of each trial series holds the
frame interval; trial times are relative to each trial, so inter-trial intervals cannot be
computed; the lick trace does not say which spout was licked; some files store non-integer pulse
rates; and the NWB session identifiers are the CNMF run timestamps, while the recording date is in
`session_start_time` and `session_description`. The trial table has no helped-trial or first-trial flag (trial_is_good equals a correct or incorrect response), and the side licked on 16 Hz trials cannot be recovered.


In [ ]:
print("runtime %.1f min, peak memory %.1f GB" % ((time.time() - T0) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))